# Demonstração da entrega — Tech Challenge Fase 2

Este notebook percorre a entrega **sem executar nada que altere estado**: ele valida assinaturas, lê artefatos congelados e exibe resultados já produzidos.

Nada aqui treina modelo, reabre seleção, altera o limiar, consulta o holdout ou faz chamada de rede.

Os comandos de reprodução metodológica — bateria genética, busca aleatória, baseline e providers reais — ficam deliberadamente fora deste notebook. Eles são caros e sobrescreveriam evidência congelada.

**Aviso:** resultado acadêmico e experimental. Os modelos não foram validados para uso clínico e não devem ser usados para diagnóstico, tratamento ou decisão médica.

## 1. Ambiente e integridade da entrega

A validação é somente leitura: confere hashes de documentos, artefatos e figuras contra o manifesto assinado.

In [ ]:
from pathlib import Path
import json, os, sys

os.chdir(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, "src")

from tech_challenge_fase2.deliverable import validate_deliverable

resultado = validate_deliverable()
reprovados = [c for c in resultado["checks"] if not c["passed"]]
print(f"validação da entrega: {'APROVADA' if resultado['passed'] else 'REPROVADA'}")
print(f"verificações: {len(resultado['checks'])} | reprovadas: {len(reprovados)}")
for c in reprovados:
    print("  ", c)

## 2. Tabela mestre

As nove origens avaliadas no holdout. `logistic_regression__random_search` foi o vencedor global, congelado **antes** do teste final.

In [ ]:
import csv

with open("artifacts/final_summary/model_results.csv", encoding="utf-8") as f:
    linhas = list(csv.DictReader(f))

colunas = ["family_label", "method_label", "recall_test", "f1_test", "roc_auc_test", "false_negatives"]
titulos = ["família", "método", "recall", "F1", "ROC-AUC", "FN"]
print(" ".join(f"{t:>22}" if i < 2 else f"{t:>10}" for i, t in enumerate(titulos)))
for linha in linhas:
    campos = []
    for i, c in enumerate(colunas):
        valor = linha[c]
        if i < 2:
            campos.append(f"{valor:>22}")
        elif c == "false_negatives":
            campos.append(f"{int(valor):>10}")
        else:
            campos.append(f"{float(valor):>10.4f}")
    print(" ".join(campos))

## 3. Comparativo original versus otimizado

O algoritmo genético reduziu falsos negativos em duas das três famílias. Não houve superioridade universal: o KNN não melhorou recall e o ROC-AUC caiu em RF e KNN.

In [ ]:
final = json.loads(Path("artifacts/final_evaluation/final_test_results.json").read_text(encoding="utf-8"))
print("nova otimização executada:", final["new_optimization_performed"])
print("seleção reaberta        :", final["selection_reopened"])

incerteza = json.loads(Path("artifacts/final_evaluation/uncertainty_results.json").read_text(encoding="utf-8"))
print("\nIncerteza registrada para:", ", ".join(sorted(incerteza.keys()))[:200])

### Figuras revisadas

In [ ]:
from IPython.display import Image, display

for nome in ["01_recall_baseline_vs_ga.png", "02_falsos_negativos_baseline_vs_ga.png", "05_intervalos_recall.png"]:
    print(nome)
    display(Image(filename=f"reports/figures/final_presentation/{nome}"))

## 4. Camada LLM segura

A LLM agregada recebe apenas resultados experimentais — nunca registros individuais. A avaliação recalcula factualidade, segurança e cinco dimensões de forma determinística, sem provider e sem rede.

In [ ]:
from tech_challenge_fase2.llm.engine import evaluate_existing_output

avaliacao = evaluate_existing_output()
print("aprovado:", avaliacao["approved"], "| nota geral:", avaliacao["overall_score"])

fact = json.loads(Path("artifacts/llm_evaluation/factuality_report.json").read_text(encoding="utf-8"))
seg = json.loads(Path("artifacts/llm_evaluation/safety_report.json").read_text(encoding="utf-8"))
print("verificações factuais:", fact.get("total_checks", len(fact.get("checks", []))), "| aprovada:", fact["passed"])
print("segurança aprovada   :", seg["passed"])

## 5. Explicação individual desidentificada

O enunciado pede explicações dos **diagnósticos produzidos pelos modelos**, não apenas dos agregados. O contrato `3.0` explica uma classificação individual usando o pipeline congelado.

A LLM recebe classe, probabilidade, limiar e cinco sinais de influência. Não recebe ID, índice, diagnóstico real, alvo nem os 30 valores brutos.

In [ ]:
saida = json.loads(Path("artifacts/llm_individual_explanation/individual_output.json").read_text(encoding="utf-8"))
texto = json.dumps(saida, ensure_ascii=False)

for proibido in ["patient_id", "raw_features", "ground_truth"]:
    assert proibido not in texto, f"campo individual vazou: {proibido}"
print("nenhum campo individual proibido presente na saída\n")

manifesto = json.loads(Path("artifacts/llm_individual_explanation/individual_explanation_manifest.json").read_text(encoding="utf-8"))
print("contrato:", manifesto["contract_version"], "| provider:", manifesto["provider"], "| aprovado:", manifesto["approved"])
print("\nbarreiras de privacidade verificadas:")
for chave, valor in manifesto["privacy"].items():
    print(f"  {chave:<28} {valor}")

## 6. Escalabilidade automática e monitoramento

A camada de serviço executa o modelo congelado sob demanda variável. A política de dimensionamento é uma função pura do backlog, testável sem relógio nem threads.

In [ ]:
from tech_challenge_fase2.serving.autoscaling import AutoscalerState, AutoscalingPolicy

politica = AutoscalingPolicy(min_workers=1, max_workers=4)
estado = AutoscalerState(policy=politica)

print(f"{'ciclo':>6} {'backlog':>8} {'decisão':>16} {'workers':>8}")
for ciclo, backlog in enumerate([2, 3, 30, 28, 22, 6, 1]):
    decisao = estado.observe(backlog=backlog, now=float(ciclo))
    print(f"{ciclo:>6} {backlog:>8} {decisao.action:>16} {estado.workers:>8}")

### Resultado medido

O relatório versionado compara pool fixo mínimo e pool autoescalável sob exatamente a mesma sequência de chegadas.

In [ ]:
relatorio = json.loads(Path("artifacts/scalability/scalability_report.json").read_text(encoding="utf-8"))

print(f"CPUs: {relatorio['environment']['available_cpus']} | threads de BLAS por worker: {relatorio['environment']['blas_threads_per_worker']}\n")
print(f"{'cenário':>22} {'p95 (ms)':>10} {'req/s':>10} {'workers':>9}")
for c in relatorio["scenarios"]:
    print(f"{c['label']:>22} {c['latency']['p95_ms']:>10.1f} {c['throughput_requests_per_second']:>10.1f} {c['max_workers_used']:>9}")

print(f"\nredução de p95: {relatorio['comparison']['p95_latency_reduction_factor']:.2f}x")
print(f"ganho de vazão : {relatorio['comparison']['throughput_gain_factor']:.2f}x")

print("\nEscalar réplicas tem um limiar de custo por pedido:")
for s in relatorio["batch_size_sweep"]:
    print(f"  {s['batch_size']:>6} registros | {s['milliseconds_per_request_serial']:>5.1f} ms/pedido | {s['speedup']:>5.2f}x")

In [ ]:
display(Image(filename="reports/figures/final_presentation/07_escalabilidade_automatica.png"))

## 7. Confirmações de escopo

O manifesto final registra explicitamente o que **não** foi feito na consolidação.

In [ ]:
entrega = json.loads(Path("artifacts/final_summary/final_delivery_manifest.json").read_text(encoding="utf-8"))
for chave, valor in entrega["scope_confirmations"].items():
    print(f"  {chave:<42} {valor}")

## Limitações

- 569 registros de fonte única; apenas 42 casos malignos no holdout, então uma observação muda o recall em cerca de 2,38 pontos percentuais.
- Sem validação externa, prospectiva ou clínica.
- Intervalos de confiança amplos; o bootstrap pareado inclui zero e o McNemar tem poucos discordantes.
- Uma seed oficial reproduz, mas não estima a variabilidade entre execuções completas do GA.
- A medição de escalabilidade depende do hardware e não deve ser citada como característica do modelo.

O detalhamento está em `docs/limitacoes_e_validade.md`.